# Different priors in ARCHS4 model building with CLAMP

💡 **Environment:** `clamp-analyses`  

## Load libraries

In [5]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    REPO_PATH <- "/home/msubirana/Documents/pivlab/CLAMP" 
    remotes::install_local(REPO_PATH, force = TRUE, dependencies = FALSE)
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)
library(hdf5r)
library(biomaRt)

source(here("config.R"))

set.seed(123)

## Output directory

In [7]:
output_dir <- config$ARCHS4$DATASET_FOLDER
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

data_path <- here::here('data/archs4')
dir.create(data_path, showWarnings = FALSE, recursive = TRUE)

In [ ]:
# meta
meta <- readRDS(file.path(output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples <- meta$n_samples

archs4_genes <- meta$gene_symbols_thin

all_samples <- readRDS(file.path(output_dir, "all_samples.rds"))
sample_names <- all_samples[seq_len(n_samples)]

# fbm
fbm_file  <- file.path(output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples,
  backingfile = output_file ,
  create_bk   = FALSE,
)

# svd
archs4_svdRes <- readRDS(file.path(output_dir, "svd.rds"))

In [5]:
archs4_baseRes <- readRDS(file.path(output_dir, "archs4_baseRes.rds"))

In [ ]:
CLAMP_K_archs4 <- readRDS(file.path(output_dir, "CLAMP_K_archs4.rds"))

## Prepare pathway priors

In [ ]:
# # run localy since server not have internet access
# KEGG_list <- list(
#   KEGG = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=KEGG_2021_Human")
# )

# for(lib in names(KEGG_list)) {
#   names(KEGG_list[[lib]]) <- paste0(lib, "_", names(KEGG_list[[lib]]))
# }

# KEGG_pathMat <- gmtListToSparseMat(KEGG_list)
# saveRDS(KEGG_pathMat, file = file.path(data_path, "KEGG_pathMat.rds"))

# BP_list <- list(
#   BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
# )

# for(lib in names(BP_list)) {
#   names(BP_list[[lib]]) <- paste0(lib, "_", names(BP_list[[lib]]))
# }

# BP_pathMat <- gmtListToSparseMat(BP_list)
# saveRDS(BP_pathMat, file = file.path(data_path, "BP_pathMat.rds"))

# GTEx_Tissues_list <- list(
#   GTEx_Tissues = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GTEx_Tissues_V8_2023")
# )

# for(lib in names(GTEx_Tissues_list)) {
#   names(GTEx_Tissues_list[[lib]]) <- paste0(lib, "_", names(GTEx_Tissues_list[[lib]]))
# }

# GTEx_Tissues_pathMat <- gmtListToSparseMat(GTEx_Tissues_list)
# saveRDS(GTEx_Tissues_pathMat, file = file.path(data_path, "GTEx_Tissues_pathMat.rds"))


# CellMarker_list <- list(
#   CellMarker = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=CellMarker_2024")
# )

# for(lib in names(CellMarker_list)) {
#   names(CellMarker_list[[lib]]) <- paste0(lib, "_", names(CellMarker_list[[lib]]))
# }

# CellMarker_pathMat <- gmtListToSparseMat(CellMarker_list)
# saveRDS(CellMarker_pathMat, file = file.path(data_path, "CellMarker_pathMat.rds"))

Auto-detected name: KEGG_2021_Human

Using cached file for KEGG_2021_Human



Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

Auto-detected name: GTEx_Tissues_V8_2023

Using cached file for GTEx_Tissues_V8_2023

Auto-detected name: CellMarker_2024

Using cached file for CellMarker_2024



## CLAMPfull KEGG

In [ ]:
KEGG_pathMat <- readRDS(file.path(output_dir, "KEGG_pathMat.rds"))
KEGG_matched <- getMatchedPathwayMat(KEGG_pathMat, archs4_genes)

archs4_fullRes <- CLAMPfull(
    Y = archs4_fbm_filt,
    svdres = archs4_svdRes,
    priorMat = KEGG_matched,
    clamp.base.result = archs4_baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    clamp_k = CLAMP_K_archs4
  )
  
# Fix col, row names and summary
archs4_fullRes$Z <- data.frame(archs4_baseRes$Z)
rownames(archs4_baseRes$Z) <- archs4_genes

archs4_fullRes$B <- data.frame(archs4_fullRes$B)
colnames(archs4_fullRes$B) <- sample_names

archs4_fullRes$summary <- archs4_fullRes$summary %>%
    dplyr::rename(LV = LV_index)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

saveRDS(archs4_fullRes, file = file.path(output_dir, "archs4_CLAMP_KEGG.rds"))

model_dir <- file.path(output_dir, "archs4_CLAMP_KEGG")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- archs4_fullRes$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- archs4_fullRes$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

summary <- archs4_fullRes$summary
write.csv(summary, file.path(model_dir, "summary.csv"))

There are 12486 genes in the intersection between data and prior



Removing 2409 pathways

Inverting...

done



## CLAMPfull BP

In [ ]:
BP_pathMat <- readRDS(file.path(output_dir, "BP_pathMat.rds"))
BP_matched <- getMatchedPathwayMat(BP_pathMat, archs4_genes)

archs4_fullRes <- CLAMPfull(
    Y = archs4_fbm_filt,
    svdres = archs4_svdRes,
    priorMat = BP_matched,
    clamp.base.result = archs4_baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    clamp_k = CLAMP_K_archs4
  )
  
# Fix col, row names and summary
archs4_fullRes$Z <- data.frame(archs4_baseRes$Z)
rownames(archs4_baseRes$Z) <- archs4_genes

archs4_fullRes$B <- data.frame(archs4_fullRes$B)
colnames(archs4_fullRes$B) <- sample_names

archs4_fullRes$summary <- archs4_fullRes$summary %>%
    dplyr::rename(LV = LV_index)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

saveRDS(archs4_fullRes, file = file.path(output_dir, "archs4_CLAMP_BP.rds"))

model_dir <- file.path(output_dir, "archs4_CLAMP_BP")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- archs4_fullRes$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- archs4_fullRes$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

summary <- archs4_fullRes$summary
write.csv(summary, file.path(model_dir, "summary.csv"))

**PLIER v2 **

Warning message:
“`seed` is deprecated and ignored. Use set.seed(seed) before calling this function.”
using provided PLIERbase result

L1=111.157238320606; L2=37.0524127735352

Progress 1 / 350 | Bdiff=0.035954

Progress 2 / 350 | Bdiff=0.005951

, Number of annotated columns is 81

Progress 3 / 350 | Bdiff=0.002198

Progress 4 / 350 | Bdiff=0.001135

, Number of annotated columns is 89

Progress 5 / 350 | Bdiff=0.000741

Progress 6 / 350 | Bdiff=0.000557

, Number of annotated columns is 98

Progress 7 / 350 | Bdiff=0.000455

Progress 8 / 350 | Bdiff=0.000393

, Number of annotated columns is 99

Progress 9 / 350 | Bdiff=0.000361

Progress 10 / 350 | Bdiff=0.000327

, Number of annotated columns is 104

Progress 11 / 350 | Bdiff=0.000304

Progress 12 / 350 | Bdiff=0.000291

, Number of annotated columns is 83

Progress 13 / 350 | Bdiff=0.000276

Progress 14 / 350 | Bdiff=0.000269

, Number of annotated columns is 88

Progress 15 / 350 | Bdiff=0.000260

Progress 16 / 350

## CLAMPfull GTEx

In [ ]:
GTEx_Tissues_pathMat <- readRDS(file.path(output_dir, "GTEx_Tissues_pathMat.rds"))
GTEx_Tissues_matched <- getMatchedPathwayMat(GTEx_Tissues_pathMat, archs4_genes)

archs4_fullRes <- CLAMPfull(
    Y = archs4_fbm_filt,
    svdres = archs4_svdRes,
    priorMat = GTEx_Tissues_matched,
    clamp.base.result = archs4_baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    clamp_k = CLAMP_K_archs4
  )
  
# Fix col, row names and summary
archs4_fullRes$Z <- data.frame(archs4_baseRes$Z)
rownames(archs4_baseRes$Z) <- archs4_genes

archs4_fullRes$B <- data.frame(archs4_fullRes$B)
colnames(archs4_fullRes$B) <- sample_names

archs4_fullRes$summary <- archs4_fullRes$summary %>%
    dplyr::rename(LV = LV_index)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

saveRDS(archs4_fullRes, file = file.path(output_dir, "archs4_CLAMP_GTEx.rds"))

model_dir <- file.path(output_dir, "archs4_CLAMP_GTEx")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- archs4_fullRes$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- archs4_fullRes$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

summary <- archs4_fullRes$summary
write.csv(summary, file.path(model_dir, "summary.csv"))

## CLAMPfull CellMarker

In [ ]:
CellMarker_pathMat <- readRDS(file.path(output_dir, "CellMarker_pathMat.rds"))
CellMarker_matched <- getMatchedPathwayMat(CellMarker_pathMat, archs4_genes)

archs4_fullRes <- CLAMPfull(
    Y = archs4_fbm_filt,
    svdres = archs4_svdRes,
    priorMat = CellMarker_matched,
    clamp.base.result = archs4_baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    clamp_k = CLAMP_K_archs4
  )
  
# Fix col, row names and summary
archs4_fullRes$Z <- data.frame(archs4_baseRes$Z)
rownames(archs4_baseRes$Z) <- archs4_genes

archs4_fullRes$B <- data.frame(archs4_fullRes$B)
colnames(archs4_fullRes$B) <- sample_names

archs4_fullRes$summary <- archs4_fullRes$summary %>%
    dplyr::rename(LV = LV_index)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

saveRDS(archs4_fullRes, file = file.path(output_dir, "archs4_CLAMP_cellMarker.rds"))

model_dir <- file.path(output_dir, "archs4_CLAMP_cellMarker")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- archs4_fullRes$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- archs4_fullRes$Z
write.csv(Z, file.path(model_dir, "Z.csv"))

summary <- archs4_fullRes$summary
write.csv(summary, file.path(model_dir, "summary.csv"))